<h1 style='font-weight:bold; color:#003366; text-align:center;'>Tema 9: Análisis exploratorio de datos</h1>

<h3 style='font-weight:bold; color:#003366; text-align:center;'>Programación Analítica II</h3>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>Docente Camila Palomeque</h4>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>S2 . 2026</h4>

<h2 style='font-weight:bold; color:#003366;'>1. Qué es el análisis exploratorio y qué no es</h2>

Hay una definición muy difundida y muy pobre: "El EDA es hacer gráficos antes de modelar para ver los datos". Si fuera
eso, sería opcional.

El análisis exploratorio es **la etapa en la que se decide qué datos son usables y se deja escrito por
qué**. 

**El resultado de un EDA no es un gráfico, es una lista de decisiones justificadas.**

Lo que sí es:

- Un inventario de lo que está mal, **con la magnitud de cada cosa**. No "hay faltantes": "falta el
  11,3% del ingreso entre los independientes y el 4,3% en el resto".
- Una decisión por problema, escrita, con su motivo y su consecuencia sobre el resultado final.
- Un mapa de qué se puede y qué no se puede concluir con estos datos.

Lo que no es:

- Correr `describe()` y `heatmap` de correlaciones y pasar al modelo.
- Borrar filas hasta que no queden nulos. Cada fila que se borra es una decisión que tiene que estar justificada.


<h2 style='font-weight:bold; color:#003366;'>2. 5 pasos mínimos del EDA</h2>

| Paso | Qué se hace | Pregunta que contesta |
|---|---|---|
| 1. Estructura | Filas, columnas, tipos, rango de fechas, cardinalidad | ¿Qué es una fila acá? |
| 2. Calidad | Faltantes, valores que indican vacío (sin categoría, -1, -9999), duplicados, rangos, consistencia, claves | ¿Qué de esto es usable? |
| 3. Univariado | Una variable a la vez: centro, dispersión, forma | ¿Cómo se distribuye cada cosa? |
| 4. Bivariado | Dos variables juntas, y contra la variable de interés | ¿Qué se relaciona con qué? |
| 5. Conclusiones finales | Decisiones tomadas (imputaciones, transformaciones de escala, corte de valores, ignorar ciertas variables) | ¿Qué cambios hice y si tienen impacto en mi tarea posterior? |

<h2 style='font-weight:bold; color:#003366;'>3. Paso 1: estructura</h2>

Frente a una tabla nueva, hay cinco cosas que se miran siempre y en este orden:
0. `info`: Nos da un panorama general de tipos de dato, cantidad de filas y columnas, nombres de columnas y si existen faltantes.
1. `shape`: cuántas filas y cuántas columnas.
2. `dtypes`: qué tipo tiene cada columna. Chequeo: `object` significa texto, y
   una columna numérica o de fecha quede como texto nos genera problemas a futuro.
3. **Qué es una fila.** Un movimiento, un cliente, un crédito, un pago.
4. El **rango de fechas**, si hay fechas. Sirve para detectar datos del futuro y períodos incompletos.
5. La **cardinalidad** de cada categórica con `nunique()`: cuántos valores distintos tiene. Una columna
   que debería tener 4 categorías y tiene 15 está sucia, y todavía no miramos ni un valor.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from pathlib import Path

pio.templates.default = "plotly_white"
COLORES = ["#003366", "#2E86AB", "#BB3300", "#7A9E9F", "#E8A33D", "#5C4D7D"]


def ruta_datos(nombre_archivo):
    candidatas = [
        Path("../Datos") / nombre_archivo,
        Path("Datos") / nombre_archivo,
        Path("/content/Datos") / nombre_archivo,
        Path(nombre_archivo),
    ]
    for candidata in candidatas:
        if candidata.exists():
            return str(candidata)
    raise FileNotFoundError(f"No se encontró {nombre_archivo}.")


def describir_estructura(datos, columna_fecha=None):
    print(datos.info())
    print(f"\nFilas: {len(datos):,}   Columnas: {datos.shape[1]}")
    print(f"Memoria: {datos.memory_usage(deep=True).sum() / 1e6:.2f} MB")

    resumen = pd.DataFrame({
        "tipo": datos.dtypes.astype(str),
        "nulos": datos.isna().sum(),
        "nulos_pct": (datos.isna().mean() * 100).round(2),
        "distintos": datos.nunique(dropna=True),
    })
    print("\n" + resumen.to_string())

    if columna_fecha is not None:
        fechas = pd.to_datetime(datos[columna_fecha], errors="coerce")
        print(f"\nRango de {columna_fecha}: {fechas.min().date()} a {fechas.max().date()}")
        print(f"Fechas ilegibles: {fechas.isna().sum()}")


transacciones = pd.read_csv(ruta_datos("transacciones_personales.csv"), parse_dates=["fecha"])
describir_estructura(transacciones, columna_fecha="fecha")

**Algunas conclusiones** 354 movimientos de una cuenta personal, entre enero de 2025 y junio de
2026. Una fila es un movimiento bancario.

Tres cosas para anotar antes de seguir:

- `monto` tiene 6 nulos (1,7%). Son movimientos que existen pero cuyo importe no vino. No es lo mismo
  que un movimiento de cero pesos.
- `categoria` tiene 30 nulos (8,5%), y `descripcion` tiene 141 valores distintos sobre 354 filas: hay
  comercios que se repiten, lo cual habilita clasificar por palabra clave.
- `medio_pago` tiene 4 valores distintos, que es lo esperable. `categoria` tiene 8: también razonable.
  Si tuviera 30, la mitad serían variantes de escritura.

<h2 style='font-weight:bold; color:#003366;'>4. Paso 2: calidad de los datos</h2>

Algunos de los chequeos que se deberian hacer a la hora de iniciar un analisis son:

| # | Chequeo | Qué se busca |
|---|---|---|
| 1 | Faltantes | Cuántos, y sobre todo **si faltan con patrón** |
| 2 | Centinelas | Faltantes disfrazados de dato: `-999`, `0`, `"N/D"`, `"SIN DATO"`, vacío |
| 3 | Números como texto | Separador de miles, coma decimal, símbolo de moneda |
| 4 | Unidades | Dos unidades conviviendo en la misma columna |
| 5 | Duplicados | Exactos y **parciales** (misma clave, distinto valor) |
| 6 | Rangos | Valores que el dominio no admite: edad 0, plazo 999 |
| 7 | Consistencia | Una columna que contradice a otra |
| 8 | Categóricas | Variantes de escritura y sinónimos |
| 9 | Claves | Huérfanos al combinar dos tablas |
| 10 | Campos calculados | Lo que el sistema ya calculó, verificado contra el origen |


### 4.1 Faltantes: el conteo no alcanza, hay que buscar el patrón

`isna().sum()` dice cuántos faltan. La pregunta que importa es otra: **¿faltan al azar o faltan en un
grupo en particular?** Se contesta agrupando la tasa de faltantes por otra variable.

La diferencia es enorme. Si faltan al azar, imputar con la mediana es defendible. Si faltan en un
grupo, imputar con la mediana global mete un sesgo justo en ese grupo y ese grupo puede ser el de interes.

In [ ]:
# 1. Cuántos faltan
print("Nulos por columna:")
print(transacciones.isna().sum().loc[lambda conteo: conteo > 0].to_string())

# 2. ¿Faltan al azar? Primera hipótesis: el signo del movimiento
transacciones["es_ingreso"] = transacciones["monto"] > 0

print("\nTasa de categoría faltante según el signo del movimiento:")
print(transacciones.groupby("es_ingreso")["categoria"]
      .apply(lambda columna: columna.isna().mean() * 100).round(1).to_string())

print("\nTasa de categoría faltante por medio de pago:")
print(transacciones.groupby("medio_pago")["categoria"]
      .apply(lambda columna: columna.isna().mean() * 100).round(1).to_string())

print("\nTasa de categoría faltante por año:")
print(transacciones.groupby(transacciones["fecha"].dt.year)["categoria"]
      .apply(lambda columna: columna.isna().mean() * 100).round(1).to_string())

**Interpretación.** El faltante **no es al azar**: el 9,1% de los egresos no tiene categoría y el 0% de
los ingresos. Por medio de pago las tasas van de 6,6% a 11,4%, un rango angosto que con 354 filas es
compatible con azar; por año, 7,5% contra 10,4%, lo mismo.

Es decir: el patrón está en el signo del movimiento, no en el medio de pago ni en el tiempo. Y tiene
una explicación operativa razonable: el banco clasifica automáticamente los créditos de la cuenta
(sueldo, transferencias recibidas) y deja sin clasificar parte de los débitos.

### 4.2 Falsos valores faltantes (centinelas)

Un **centinela** es un valor que un sistema usa para decir "no hay dato" sin usar un campo vacío. Es
una herencia de sistemas viejos donde una columna numérica no podía estar vacía. Los habituales:

| Centinela | Dónde aparece | Por qué es peligroso |
|---|---|---|
| `-999`, `-1`, `9999` | Campos numéricos | Se cuela en promedios y correlaciones sin hacer ruido |
| `0` | Ingresos, scores, saldos | **El peor de todos**, porque cero es un valor válido en muchas columnas |
| `"N/D"`, `"S/D"`, `"SIN DATO"`, `"-"` | Campos de texto | Convierte la columna entera en `object` |
| Texto vacío o solo espacios | Cualquiera | `isna()` no lo detecta si tiene un espacio adentro |
| `1900-01-01`, `9999-12-31` | Fechas | Rompe cualquier cálculo de antigüedad |

**El chequeo:** para cada columna numérica, mirar el mínimo, el máximo
y los valores más frecuentes, y compararlos contra el rango que el negocio admite. Ninguna de esas tres
cosas la muestra `isna()`.

### 4.3 Números que vinieron como texto

`ingreso_declarado` es la columna más importante de la tabla y llegó como texto. Fijate cómo se ve.

In [ ]:
import numpy as np

transacciones["ingreso_declarado"] = np.random.choice(
    [   "$ 45.000", "UYU 40.000", "28000", "$145,000", "70000", "85 000", "90000", "-999999", "USD 1000",
        "80.000,00", "115,000.00", "N/D", "SIN DATO", "", "38000 pesos", "45k", "67000", '90000', "USD 1500"],
    size=len(transacciones))

print("Diez valores tal como vinieron:")
print(transacciones["ingreso_declarado"].sample(10, random_state=3).to_string())

print("\nNulos que declara la columna:", transacciones["ingreso_declarado"].isna().sum())

# La tentación: convertir a número de un saque y seguir con la vida
ingreso_apurado = pd.to_numeric(transacciones["ingreso_declarado"], errors="coerce")

print("\nSi aplico pd.to_numeric(errors='coerce') para pasarlo a numero sin ninguna limpieza, obtengo:" )
print(f"Nulos antes: {transacciones['ingreso_declarado'].isna().mean():.1%}")
print(f"Nulos después: {ingreso_apurado.isna().mean():.1%}")
print(f"Filas que se volvieron NaN: {ingreso_apurado.isna().sum():,}")
print(f"Valores menores o iguales a cero que siguen en la variable: {(ingreso_apurado <= 0).sum()}")

**Acá se pierde la mitad de un proyecto y nadie se entera.** `errors="coerce"` significa "lo que no
puedas convertir, ponelo como faltante". 

Esto genera muchos nulos que son falsos, por ejemplo en este caso:

- `"45.000,00"` y `"$ 45.000"` no son números **para Python**, pero sí son ingresos perfectamente
  informados. Ese dato no falta: está escrito en otro formato.
- Al mismo tiempo, `"-999"` y `"0"` **sí** se convierten en números y sobreviven lo que valida valores que no son ingresos. 

O sea que la conversión apurada hace las dos cosas mal a la vez: borra datos buenos y conserva datos
inventados.

In [ ]:
def texto_a_numero(valor):
    """Convierte a número un importe escrito con los dos formatos del export.

    Cuando hay coma, el punto es separador de miles. "45.000" sin coma se interpreta
    como cuarenta y cinco mil, no como 45 con tres decimales.

    Elimina marcas de moneda, espacios y la palabra "pesos". Convierte "k" a miles. 
    Convierte a nan valores vacíos, "N/D" y "SIN DATO" y valores no numéricos.
    """

    texto = str(valor).strip().replace("$", "").replace(" ", "").replace("UYU", "").replace("pesos", "").replace("k", "000").replace("USD", "").strip()
    if texto == "" or texto.upper() in ("N/D", "SIN DATO"):
        return np.nan
    if "," in texto:
        texto = texto.replace(".", "").replace(",", ".")
    elif texto.count(".") == 1 and len(texto.split(".")[1]) == 3:
        texto = texto.replace(".", "")
    try:
        return float(texto)
    except ValueError:
        return np.nan


columna = transacciones["ingreso_declarado"]

print("Nulos después de cada paso de la limpieza")
print(f"  0. Inicial: {columna.isna().mean():6.2%}")
print(f"  1. to_numeric a secas (lo que NO hay que hacer): "
      f"{pd.to_numeric(columna, errors='coerce').isna().mean():6.2%}")

ingreso = columna.apply(texto_a_numero)
transacciones["ingreso_declarado_limpio"] = ingreso
print(f"  2. formato de miles y decimales resuelto: {ingreso.isna().mean():6.2%}")
print(f"     mínimo de la columna en este punto: {ingreso.min():,.0f}")

ingreso = ingreso.mask(ingreso <= 0)
print(f"  3. Valores imposibles numéricos (0 y -999) a NaN: {ingreso.isna().mean():6.2%}")

print(f"\nIngreso mediano recuperado: $ {ingreso.median():,.0f}")
print(f"Filas con ingreso usable: {ingreso.notna().sum():,} de {len(transacciones):,}")

In [ ]:
transacciones


### 4.4 Unidades distintas conviviendo en la misma columna

Ojo que al visualizar ingreso declarado vemos que algunos integrantes declararon su ingreso en dolares y otros en pesos uruguayos, eso es importante normalizarlo en la columna limpia

### 4.5 Duplicados: exactos y parciales

Dos problemas con el mismo nombre y con dificultad muy distinta.

In [ ]:
# Export de solicitudes de crédito: 100 clientes distintos más los duplicados que trae todo export.

generador_cartera = np.random.default_rng(2026)
N_CLIENTES = 100

SECTORES = ["Comercio", "Industria", "Servicios", "Construcción", "Agro"]
DEPARTAMENTOS = ["Montevideo", "Canelones", "Maldonado", "Salto", "Colonia", "Rivera"]

VARIANTES_SECTOR = {
    "Comercio":     [" Comercio ", "COMERCIO", "comercio", "Comercio"],
    "Industria":    ["Industria", "INDUSTRIA ", "industria", " industria"],
    "Servicios":    ["Servicios", "servicios", "SERVICIOS", "Servicio"],
    "Construcción": ["Construcción", "Construccion", "CONSTRUCCION", "construccion "],
    "Agro":         ["Agro", "agro", "AGRO", "Agropecuario"],
}

id_cliente = np.arange(101, 101 + N_CLIENTES)

# Ingreso mensual en pesos uruguayos, con cola a la derecha
ingreso = np.round(generador_cartera.lognormal(mean=10.85, sigma=0.42, size=N_CLIENTES) / 1000) * 1000
ingreso = np.clip(ingreso, 22000, 400000)

# El monto solicitado acompaña al ingreso, con ruido
monto_solicitado = np.round(ingreso * generador_cartera.uniform(0.8, 2.4, N_CLIENTES) / 5000) * 5000

plazo_meses = generador_cartera.choice([6, 12, 18, 24, 36, 48, 60],
                                       size=N_CLIENTES,
                                       p=[0.08, 0.20, 0.12, 0.25, 0.20, 0.08, 0.07])

edad = generador_cartera.integers(19, 72, size=N_CLIENTES)
antiguedad_laboral = np.minimum(generador_cartera.integers(0, 35, size=N_CLIENTES), edad - 18)

# Tasa anual: la mayoría entre 24% y 58%
tasa_pactada = np.round(generador_cartera.uniform(0.24, 0.58, N_CLIENTES), 2)

cuota_sistema = np.round(monto_solicitado * (1 + tasa_pactada * plazo_meses / 12) / plazo_meses / 100) * 100

# La mora depende del esfuerzo de pago y de la antigüedad laboral
esfuerzo = cuota_sistema / ingreso
probabilidad_mora = np.clip(0.04 + 1.1 * esfuerzo - 0.004 * antiguedad_laboral, 0.02, 0.85)
mora_90_dias = (generador_cartera.random(N_CLIENTES) < probabilidad_mora).astype(int)

sector_base = generador_cartera.choice(SECTORES, size=N_CLIENTES,
                                       p=[0.28, 0.20, 0.24, 0.18, 0.10])
sector_empleo = [generador_cartera.choice(VARIANTES_SECTOR[s]) for s in sector_base]

departamento_base = generador_cartera.choice(DEPARTAMENTOS, size=N_CLIENTES,
                                             p=[0.42, 0.18, 0.12, 0.12, 0.09, 0.07])
estilos = generador_cartera.choice(["minuscula", "titulo_espacio", "mayuscula", "titulo"],
                                   size=N_CLIENTES)
departamento = [
    d.lower() if e == "minuscula" else
    d + " " if e == "titulo_espacio" else
    d.upper() if e == "mayuscula" else d
    for d, e in zip(departamento_base, estilos)
]

# Fechas: la mitad vienen como AAAA-MM-DD y la mitad como DD/MM/AAAA
anios = generador_cartera.choice([2025, 2026], size=N_CLIENTES, p=[0.75, 0.25])
meses = np.where(anios == 2026,
                 generador_cartera.integers(1, 7, size=N_CLIENTES),
                 generador_cartera.integers(1, 13, size=N_CLIENTES))
dias = generador_cartera.integers(1, 13, size=N_CLIENTES)   # día <= 12: por eso el error es silencioso
con_barras = generador_cartera.random(N_CLIENTES) < 0.5
fecha_solicitud = [
    f"{d:02d}/{m:02d}/{a}" if barras else f"{a}-{m:02d}-{d:02d}"
    for a, m, d, barras in zip(anios, meses, dias, con_barras)
]

solicitudes = pd.DataFrame({
    "id_cliente": id_cliente,
    "monto_solicitado": monto_solicitado.astype(int),
    "plazo_meses": plazo_meses,
    "cuota_sistema": cuota_sistema.astype(int),
    "mora_90_dias": mora_90_dias,
    "tasa_pactada": tasa_pactada,
    "edad": edad,
    "antiguedad_laboral": antiguedad_laboral,
    "sector_empleo": sector_empleo,
    "departamento": departamento,
    "fecha_solicitud": fecha_solicitud,
    "ingreso": ingreso.astype(int),
})

# a) Tasas cargadas en porcentaje en vez de tanto por uno
filas_en_porcentaje = generador_cartera.choice(N_CLIENTES, size=18, replace=False)
solicitudes.loc[filas_en_porcentaje, "tasa_pactada"] = (
    solicitudes.loc[filas_en_porcentaje, "tasa_pactada"] * 100).round(0)

# b) Edades fuera del rango admitido por el negocio
solicitudes.loc[3, "edad"] = 17
solicitudes.loc[47, "edad"] = 95
solicitudes.loc[88, "edad"] = 16

# c) Plazos fuera del catálogo
solicitudes.loc[7, "plazo_meses"] = 3
solicitudes.loc[62, "plazo_meses"] = 72
solicitudes.loc[91, "plazo_meses"] = 84

# d) Antigüedad laboral inconsistente con la edad
solicitudes.loc[4, ["edad", "antiguedad_laboral"]] = [55, 60]
solicitudes.loc[26, ["edad", "antiguedad_laboral"]] = [30, 35]
solicitudes.loc[70, ["edad", "antiguedad_laboral"]] = [24, 15]

# e) Una fila cargada dos veces exactamente igual y dos clientes recargados con otros datos
fila_repetida_exacta = solicitudes.loc[[12]]

correccion_a = solicitudes.loc[[30]].copy()
correccion_a["monto_solicitado"] = correccion_a["monto_solicitado"] + 5000
correccion_a["cuota_sistema"] = correccion_a["cuota_sistema"] + 200

correccion_b = solicitudes.loc[[55]].copy()
correccion_b["monto_solicitado"] = correccion_b["monto_solicitado"] - 10000
correccion_b["cuota_sistema"] = correccion_b["cuota_sistema"] - 300

In [ ]:
solicitudes = pd.concat([solicitudes, fila_repetida_exacta, correccion_a, correccion_b],
                        ignore_index=True)

print(f"Filas del export: {len(solicitudes):,}")

exactos = solicitudes.duplicated()
print(f"Filas 100% duplicadas: {exactos.sum()}")

sin_exactos = solicitudes.drop_duplicates()
print(f"Filas después de drop_duplicates: {len(sin_exactos):,}")

# Duplicados parciales: el mismo cliente aparece dos veces con datos distintos
ids_repetidos = sin_exactos["id_cliente"].duplicated().sum()
print(f"\nIdentificadores repetidos que quedan: {ids_repetidos}")

repetidos = sin_exactos[sin_exactos["id_cliente"].duplicated(keep=False)]
print("\nLas dos versiones de esos clientes:")
print(repetidos.sort_values("id_cliente")
      [["id_cliente", "monto_solicitado", "plazo_meses", "cuota_sistema", "mora_90_dias"]]
      .head(6).to_string(index=False))


1 filas era copias exacta: doble carga del mismo formulario, no aportan información y se eliminan sin discusión. `drop_duplicates()`.

Los **2 identificadores que siguen repetidos son otro problema y no tiene solución automática.** El
mismo cliente aparece dos veces con montos y cuota distinta. Puede ser:

- una corrección posterior, y entonces vale la última
- dos solicitudes reales del mismo cliente, y entonces valen las dos
- una carga con error, y entonces vale la que sea consistente con el resto de las columnas

`drop_duplicates(subset="id_cliente")` elige la primera fila que aparece, que es la primera que el
sistema exportó, que no es un criterio: es el orden del archivo. **Si vas a quedarte con una, el
criterio tiene que estar escrito.** 

### 4.6 Valores que no pueden ser: unidades, rangos y consistencia

Tres chequeos que se parecen y son distintos:

- **Unidades:** la columna está bien pero mide dos cosas distintas en filas distintas.
- **Rangos:** el valor está afuera de lo que el dominio del negocio admite.
- **Consistencia:** el valor de una columna contradice el de otra.

In [ ]:
cartera = sin_exactos.copy()

# Unidades: una tasa de crédito al consumo no llega al 150% anual, así que
# todo lo que esté arriba de 1,5 vino en porcentaje y no en tanto por uno.
tasa = cartera["tasa_pactada"]
en_porcentaje = tasa > 1.5
print("Distribución de tasa_pactada tal como vino:")
print(tasa.describe().round(2).to_string())
print(f"\nFilas que vinieron en porcentaje: {en_porcentaje.sum()} ({en_porcentaje.mean():.1%})")
print(f"Promedio sin corregir: {tasa.mean():.2f}")
print(f"Promedio corregido: {tasa.mask(en_porcentaje, tasa / 100).mean():.2%}\n")

# Rangos: lo que el dominio no admite
RANGOS_VALIDOS = {
    "edad": (18, 90),               # no se puede firmar un crédito antes de los 18
    "antiguedad_laboral": (0, 65),  # en años
    "plazo_meses": (6, 60),         # catálogo de plazos de la empresa
}

for variable, (minimo, maximo) in RANGOS_VALIDOS.items():
    fuera = ~cartera[variable].between(minimo, maximo)
    fuera_de_rango = fuera & cartera[variable].notna()
    print(f"{variable:20s} fuera de [{minimo}, {maximo}]: {fuera_de_rango.sum():3d}  "
          f"valores: {sorted(cartera.loc[fuera_de_rango, variable].unique())[:6]}")

# Consistencia: una columna contra otra
edad_al_empezar_a_trabajar = cartera["edad"] - cartera["antiguedad_laboral"]

imposible = edad_al_empezar_a_trabajar < 0
implausible = edad_al_empezar_a_trabajar.between(0, 16, inclusive="left")

print(f"\nAntigüedad laboral mayor que la edad (imposible)      : {imposible.sum()}")
print(f"Habría empezado a trabajar antes de los 16 (implausible): {implausible.sum()}")
print(f"Edad mínima al empezar a trabajar en la tabla           : "
      f"{edad_al_empezar_a_trabajar.min():.0f} años")
print("\nTres casos imposibles:")
print(cartera.loc[imposible, ["id_cliente", "edad", "antiguedad_laboral"]].head(3).to_string(index=False))

### 4.7 Texto sin normalizar: categorías y fechas

Dos columnas de texto, dos problemas parecidos. En las **categóricas** hay que contar los valores
distintos antes de agrupar, siempre. Y en las **fechas** hay que mirar el rango después de parsear,
siempre.

In [ ]:
cartera

In [ ]:
SINONIMOS_SECTOR = {
    "construccion": "construcción",
    "agropecuario": "agro",
    "agricultura": "agro",
    "comercio minorista": "comercio",
    "comercio mayorista": "comercio",
    "servicio": "servicios"
}

print(f"sector_empleo: {cartera['sector_empleo'].nunique()} valores distintos")
print(cartera['sector_empleo'].value_counts().to_string())

sector_normalizado = cartera['sector_empleo'].str.strip().str.lower()
print(f"\nDespués de strip() y lower(): {sector_normalizado.nunique()} valores")
print(sorted(sector_normalizado.unique()))

sector_normalizado = sector_normalizado.replace(SINONIMOS_SECTOR).str.capitalize()
print(f"\nDespués del mapa de sinónimos: {sector_normalizado.nunique()} valores")
print(sector_normalizado.value_counts().to_string())

cartera["sector"] = sector_normalizado

In [ ]:
# Normalizo departamentos: elimino espacios y pongo la primera letra en mayúscula y el resto en minúscula
cartera["departamento_norm"] = cartera["departamento"].str.strip().str.title()

print(f"\ndepartamento: {cartera['departamento'].nunique()} valores distintos, "
      f"{cartera['departamento_norm'].nunique()} después de normalizar")
print(cartera["departamento_norm"].value_counts().to_string())

In [ ]:
# Las fechas vinieron en dos formatos: AAAA-MM-DD y DD/MM/AAAA
print("Seis fechas tal como vinieron:")
print(cartera["fecha_solicitud"].sample(6, random_state=5).tolist())

# Opción A: parsear sin decirle nada. No falla.
fechas_sin_avisar = pd.to_datetime(cartera["fecha_solicitud"], format="mixed")

# Opción B: avisarle que en el formato con barras el día va primero
fechas_bien = pd.to_datetime(cartera["fecha_solicitud"], format="mixed", dayfirst=True)

print(f"\nSin dayfirst : {fechas_sin_avisar.min().date()} a {fechas_sin_avisar.max().date()}")
print(f"Con dayfirst : {fechas_bien.min().date()} a {fechas_bien.max().date()}")
print(f"Fechas que quedaron distintas: {(fechas_sin_avisar != fechas_bien).sum()}")

# Por qué es silencioso: con día menor o igual a 12, las dos lecturas son válidas
ambiguas = pd.Series(["05/03/2025", "11/07/2025", "2025-03-05"])
print("\nEl caso silencioso, tres fechas ambiguas:")
print("  sin dayfirst:", pd.to_datetime(ambiguas, format="mixed").dt.strftime("%d-%b-%Y").tolist())
print("  con dayfirst:", pd.to_datetime(ambiguas, format="mixed", dayfirst=True)
      .dt.strftime("%d-%b-%Y").tolist())

cartera["fecha_solicitud"] = fechas_bien

<h2 style='font-weight:bold; color:#003366;'>5. Paso 3: univariado, una variable a la vez</h2>

Recién ahora se puede mirar la distribución de cada variable. Para las numéricas, tres preguntas:
dónde está el centro, cuánto se dispersa, y qué forma tiene.

La comparación más informativa y más barata es **media contra mediana**:

- Media y mediana parecidas: distribución simétrica, el promedio describe bien al caso típico.
- Media bastante mayor que la mediana: **cola a la derecha**. Hay pocos casos muy grandes que estiran
  el promedio. Es lo normal en ingresos, montos, facturación y tiempos de pago.
- Media menor que la mediana: cola a la izquierda, mucho menos frecuente en finanzas.

La **asimetría** (`skew()`) pone un número a eso: 0 es simétrico, más de 1 es cola a la derecha marcada.

In [ ]:
VARIABLES_NUMERICAS = ["ingreso", "monto_solicitado", "antiguedad_laboral", "edad"]

perfil = pd.DataFrame({
    "casos": cartera[VARIABLES_NUMERICAS].notna().sum(),
    "media": cartera[VARIABLES_NUMERICAS].mean(),
    "mediana": cartera[VARIABLES_NUMERICAS].median(),
    "desvio": cartera[VARIABLES_NUMERICAS].std(),
    "asimetria": cartera[VARIABLES_NUMERICAS].skew(),
    "p95": cartera[VARIABLES_NUMERICAS].quantile(0.95),
    "maximo": cartera[VARIABLES_NUMERICAS].max(),
})
perfil["media_sobre_mediana"] = perfil["media"] / perfil["mediana"]
print(perfil.round(2).to_string())

print(f"\nAsimetría del ingreso: {cartera['ingreso'].skew():.2f}")
print(f"Asimetría del logaritmo del ingreso: {np.log(cartera['ingreso']).skew():.2f}")

In [ ]:
media_ingreso = cartera["ingreso"].mean()
mediana_ingreso = cartera["ingreso"].median()

fig = px.histogram(cartera, x="ingreso", nbins=30,
                   color_discrete_sequence=["#003366"])
fig.add_vline(x=media_ingreso, line_dash="dash", line_color="#BB3300",
              annotation_text=f"Media $ {media_ingreso:,.0f}", annotation_position="top right")
fig.add_vline(x=mediana_ingreso, line_dash="solid", line_color="#E8A33D",
              annotation_text=f"Mediana $ {mediana_ingreso:,.0f}", annotation_position="top left")
fig.update_traces(marker_line_color="white", marker_line_width=1)
fig.update_layout(
    title="Distribución del ingreso mensual declarado, con media y mediana marcadas",
    xaxis_title="Ingreso mensual declarado ($)",
    yaxis_title="Cantidad de solicitudes",
    bargap=0.05, height=450, showlegend=False)
fig.show()


In [ ]:
# Caja y bigotes: centro, dispersión y puntos extremos de cada variable monetaria
fig = go.Figure()
fig.add_trace(go.Box(x=cartera["ingreso"], name="Ingreso declarado",
                     marker_color="#003366", boxmean=True))
fig.add_trace(go.Box(x=cartera["monto_solicitado"], name="Monto solicitado",
                     marker_color="#2E86AB", boxmean=True))
fig.update_layout(
    title="Recorrido, cuartiles y valores extremos del ingreso y del monto solicitado",
    xaxis_title="Pesos ($)",
    height=420, showlegend=False)
fig.show()


In [ ]:
# Dos variables de perfil del solicitante, una al lado de la otra
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Edad del solicitante (años)",
                                    "Antigüedad laboral (años)"))
fig.add_trace(go.Histogram(x=cartera["edad"], nbinsx=20, marker_color="#003366",
                           marker_line_color="white", marker_line_width=1), row=1, col=1)
fig.add_trace(go.Histogram(x=cartera["antiguedad_laboral"], nbinsx=20, marker_color="#7A9E9F",
                           marker_line_color="white", marker_line_width=1), row=1, col=2)
fig.update_yaxes(title_text="Cantidad de solicitudes", row=1, col=1)
fig.update_layout(
    title="Cantidad de solicitudes por edad y por antigüedad laboral",
    height=420, showlegend=False, bargap=0.05)
fig.show()


In [ ]:
# Y las categóricas: cómo se reparten los casos y si hay categorías con muy pocos
CATEGORICAS = ["sector", "departamento_norm"]
for variable in CATEGORICAS:
    frecuencias = cartera[variable].value_counts(normalize=True, dropna=False) * 100
    print(f"\n{variable} ({cartera[variable].nunique()} categorías)")
    print(frecuencias.round(1).to_string())

CASOS_MINIMOS = 50
conteo_departamentos = cartera["departamento_norm"].value_counts()
chicas = conteo_departamentos[conteo_departamentos < CASOS_MINIMOS]
print(f"\nCategorías de departamento_norm con menos de {CASOS_MINIMOS} casos:")
print(chicas.to_string())



In [ ]:
# Frecuencia de cada categoría, ordenada de mayor a menor
conteo_sector = cartera["sector"].value_counts().sort_values()
conteo_depto = cartera["departamento_norm"].value_counts().sort_values()

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Sector de empleo", "Departamento"))
fig.add_trace(go.Bar(y=conteo_sector.index, x=conteo_sector.values, orientation="h",
                     marker_color="#003366", text=conteo_sector.values,
                     textposition="outside"), row=1, col=1)
fig.add_trace(go.Bar(y=conteo_depto.index, x=conteo_depto.values, orientation="h",
                     marker_color="#2E86AB", text=conteo_depto.values,
                     textposition="outside"), row=1, col=2)
fig.update_xaxes(title_text="Cantidad de solicitudes")
fig.update_layout(
    title="Cantidad de solicitudes en cada categoría de sector y de departamento",
    height=430, showlegend=False)
fig.show()


<h2 style='font-weight:bold; color:#003366;'>6. Valores atípicos: tres criterios y cuatro decisiones</h2>

Un **valor atípico** es una observación que se aparta mucho del resto. Antes de cualquier fórmula, la
distinción que importa:

> **Un valor extremo legítimo no es un error.** En finanzas los extremos suelen ser lo más importante
> de la tabla: el cliente que concentra el 20% de la facturación, el crédito que se llevó la pérdida del
> trimestre, el día que se movió el tipo de cambio.

Si un banco elimina los créditos grandes porque son atípicos, se queda con una cartera que no existe.
La pregunta nunca es "¿es grande?", es **"¿es real?"**, y eso se contesta con las otras columnas.

Hay tres criterios habituales para señalar candidatos, y dan respuestas distintas sobre los mismos
datos. Eso no es un defecto: es el punto.

| Criterio | Cómo se calcula | Cuándo conviene |
|---|---|---|
| Rango intercuartílico | Fuera de `[Q1 - k*IQR, Q3 + k*IQR]`, con `k` = 1,5 o 3 | El más usado. No se deja arrastrar por los extremos |
| Desvíos estándar | Más de 3 desvíos de la media | Solo sirve si la distribución es simétrica |
| Percentiles extremos | Fuera del 1% y el 99% | Marca siempre el mismo porcentaje de casos, por construcción |

In [ ]:
def contar_atipicos(serie, k_iqr=1.5, desvios=3, percentil=0.01):
    """Cuenta los atípicos de una serie según los tres criterios habituales."""
    valores = serie.dropna()
    q1, q3 = valores.quantile([0.25, 0.75])
    rango = q3 - q1
    por_iqr = ((valores < q1 - k_iqr * rango) | (valores > q3 + k_iqr * rango)).sum()

    puntajes = (valores - valores.mean()) / valores.std()
    por_desvios = (puntajes.abs() > desvios).sum()

    limite_bajo, limite_alto = valores.quantile([percentil, 1 - percentil])
    por_percentil = ((valores < limite_bajo) | (valores > limite_alto)).sum()

    return pd.Series({"casos": len(valores), "por_iqr": por_iqr,
                      "por_desvios": por_desvios, "por_percentil": por_percentil})


comparacion = pd.DataFrame({
    "ingreso": contar_atipicos(cartera["ingreso"]),
    "monto_solicitado": contar_atipicos(cartera["monto_solicitado"]),
}).T
display(comparacion)

# Los diez ingresos más altos, mirados junto a las otras columnas
print("\nLos ingresos más altos, con sus otras columnas:")
display(cartera.nlargest(6, "ingreso")[["ingreso", "monto_solicitado", "antiguedad_laboral",
                                      "sector", "mora_90_dias"]]
      .round(2))

In [ ]:
# Dónde caen los límites del criterio del rango intercuartílico sobre la distribución real
serie = cartera["ingreso"].dropna()
q1, q3 = serie.quantile([0.25, 0.75])
iqr = q3 - q1
limite_inferior, limite_superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr

fig = go.Figure()
fig.add_trace(go.Box(y=serie, name="Ingreso", boxpoints="all", jitter=0.5, pointpos=-1.8,
                     marker_color="#003366", marker_size=5, line_color="#003366"))
fig.add_hline(y=limite_superior, line_dash="dash", line_color="#BB3300",
              annotation_text=f"Q3 + 1,5·IQR = $ {limite_superior:,.0f}")
fig.add_hline(y=limite_inferior, line_dash="dash", line_color="#BB3300",
              annotation_text=f"Q1 - 1,5·IQR = $ {limite_inferior:,.0f}")
fig.update_layout(
    title="Ingreso de cada solicitud y límites del criterio de rango intercuartílico",
    yaxis_title="Ingreso mensual declarado ($)",
    height=500, showlegend=False)
fig.show()


<h2 style='font-weight:bold; color:#003366;'>7. Paso 4: bivariado, dos variables a la vez</h2>

Recién en el paso 4 se miran relaciones. Qué corresponde hacer depende de los tipos:

| Variable A | Variable B | Qué se usa |
|---|---|---|
| Numérica | Numérica | Correlación y gráfico de dispersión |
| Numérica | Categórica | Mediana por grupo, boxplot |
| Categórica | Categórica | Tabla de contingencia con porcentajes |
| Cualquiera | Objetivo binario | **Tasa del evento por tramo**, que es la sección que viene |

Para dos numéricas hay dos correlaciones y no dan lo mismo:

- **Pearson** mide relación lineal sobre los valores. Un solo punto extremo la mueve mucho.
- **Spearman** mide relación monótona sobre los **puestos** (el más alto, el segundo, el tercero). No le
  importa la magnitud, solo el orden, así que los extremos no la arrastran.

Con distribuciones asimétricas y atípicos **conviene mirar Spearman**, o
las dos y comparar.

In [ ]:
PARA_CORRELACION = ["ingreso", "monto_solicitado", "edad",
                    "antiguedad_laboral", "mora_90_dias"]

pearson = cartera[PARA_CORRELACION].corr(method="pearson")
spearman = cartera[PARA_CORRELACION].corr(method="spearman")

print("Pearson (relación lineal):")
display(pearson.round(2))

print("\nDiferencia absoluta entre Pearson y Spearman:")
display((pearson - spearman).abs().round(3))

print(f"\nIngreso contra monto solicitado")
print(f"Pearson: {pearson.loc['ingreso', 'monto_solicitado']:.3f}")
print(f"Spearman: {spearman.loc['ingreso', 'monto_solicitado']:.3f}")

In [ ]:
# Matriz de correlación de Spearman entre las variables numéricas
fig = px.imshow(spearman.round(2), text_auto=True, aspect="auto",
                color_continuous_scale="RdBu_r", zmin=-1, zmax=1)
fig.update_layout(
    title="Coeficiente de correlación de Spearman entre pares de variables numéricas",
    height=520, coloraxis_colorbar_title="ρ")
fig.show()


In [ ]:
# Dos numéricas: cada punto es una solicitud
fig = px.scatter(cartera, x="ingreso", y="monto_solicitado",
                 color=cartera["mora_90_dias"].map({0: "Sin mora", 1: "Con mora a 90 días"}),
                 size="plazo_meses", size_max=16,
                 hover_data=["id_cliente", "sector", "departamento_norm", "plazo_meses"],
                 color_discrete_map={"Sin mora": "#2E86AB", "Con mora a 90 días": "#BB3300"})
fig.update_layout(
    title="Monto solicitado frente a ingreso declarado, según mora a 90 días (tamaño = plazo en meses)",
    xaxis_title="Ingreso mensual declarado ($)",
    yaxis_title="Monto solicitado ($)",
    legend_title_text="", height=520)
fig.show()


In [ ]:
# Numérica contra categórica: distribución del ingreso dentro de cada sector
orden_sectores = (cartera.groupby("sector")["ingreso"].median()
                  .sort_values(ascending=False).index.tolist())

fig = px.box(cartera, x="sector", y="ingreso", color="sector",
             category_orders={"sector": orden_sectores},
             points="all", color_discrete_sequence=COLORES)
fig.update_layout(
    title="Distribución del ingreso declarado dentro de cada sector de empleo",
    xaxis_title="Sector de empleo",
    yaxis_title="Ingreso mensual declarado ($)",
    height=500, showlegend=False)
fig.show()


In [ ]:
# Objetivo binario contra una numérica: tasa de mora por tramo de ingreso
cartera["tramo_ingreso"] = pd.qcut(cartera["ingreso"], q=4,
                                   labels=["Q1 (más bajo)", "Q2", "Q3", "Q4 (más alto)"])

mora_por_tramo = (cartera.groupby("tramo_ingreso", observed=True)
                  .agg(solicitudes=("mora_90_dias", "size"),
                       tasa_mora=("mora_90_dias", lambda columna: columna.mean() * 100))
                  .reset_index())

mora_por_sector = (cartera.groupby("sector", observed=True)
                   .agg(solicitudes=("mora_90_dias", "size"),
                        tasa_mora=("mora_90_dias", lambda columna: columna.mean() * 100))
                   .reset_index().sort_values("tasa_mora"))

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Por cuartil de ingreso", "Por sector de empleo"))
fig.add_trace(go.Bar(x=mora_por_tramo["tramo_ingreso"].astype(str), y=mora_por_tramo["tasa_mora"],
                     marker_color="#003366",
                     text=mora_por_tramo["tasa_mora"].round(1).astype(str) + "%",
                     textposition="outside",
                     customdata=mora_por_tramo["solicitudes"],
                     hovertemplate="%{x}<br>Mora: %{y:.1f}%<br>Solicitudes: %{customdata}<extra></extra>"),
              row=1, col=1)
fig.add_trace(go.Bar(x=mora_por_sector["sector"], y=mora_por_sector["tasa_mora"],
                     marker_color="#2E86AB",
                     text=mora_por_sector["tasa_mora"].round(1).astype(str) + "%",
                     textposition="outside",
                     customdata=mora_por_sector["solicitudes"],
                     hovertemplate="%{x}<br>Mora: %{y:.1f}%<br>Solicitudes: %{customdata}<extra></extra>"),
              row=1, col=2)
fig.update_yaxes(title_text="Solicitudes con mora a 90 días (%)", row=1, col=1)
fig.update_layout(
    title="Porcentaje de solicitudes con mora a 90 días por cuartil de ingreso y por sector",
    height=460, showlegend=False)
fig.show()


In [ ]:
# Dos categóricas: composición por sector dentro de cada departamento
tabla = pd.crosstab(cartera["departamento_norm"], cartera["sector"], normalize="index") * 100

fig = px.bar(tabla.round(1), barmode="stack", color_discrete_sequence=COLORES)
fig.update_layout(
    title="Composición porcentual de los sectores de empleo dentro de cada departamento",
    xaxis_title="Departamento",
    yaxis_title="Participación sobre el total del departamento (%)",
    legend_title_text="Sector", height=480)
fig.show()


In [ ]:
# Una numérica a lo largo del tiempo: monto solicitado mes a mes
por_mes = (cartera.set_index("fecha_solicitud")
           .resample("MS")
           .agg(solicitudes=("monto_solicitado", "size"),
                monto_mediano=("monto_solicitado", "median"))
           .reset_index())

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=por_mes["fecha_solicitud"], y=por_mes["solicitudes"],
                     name="Solicitudes del mes", marker_color="#C9D6E3"),
              secondary_y=False)
fig.add_trace(go.Scatter(x=por_mes["fecha_solicitud"], y=por_mes["monto_mediano"],
                         name="Monto solicitado mediano", mode="lines+markers",
                         line=dict(color="#BB3300", width=3)),
              secondary_y=True)
fig.update_yaxes(title_text="Cantidad de solicitudes", secondary_y=False)
fig.update_yaxes(title_text="Monto solicitado mediano ($)", secondary_y=True)
fig.update_layout(
    title="Cantidad de solicitudes y monto solicitado mediano por mes",
    xaxis_title="Mes de la solicitud", height=460)
fig.show()


<h2 style='font-weight:bold; color:#003366;'>8. Comparaciones injustas</h2>

El planteo. Queremos saber si la cartera está mejorando o empeorando. Medimos, para cada grupo de
clientes según el mes en que entraron, qué porcentaje **se atrasó alguna vez**. Y armamos la serie.

El problema: un cliente que entró hace un año tuvo doce meses para atrasarse. Uno que entró el mes
pasado tuvo uno. Como el indicador es "alguna vez", los clientes nuevos **siempre** van a parecer
mejores, sin que nadie haya mejorado nada.

> **La serie no mide calidad. Mide antigüedad.**

Eso se llama **censura**: los casos recientes están observados durante menos tiempo, así que su evento
todavía no tuvo oportunidad de ocurrir. Y una **cohorte** es el conjunto de casos que entraron en el
mismo período, que es la forma correcta de agruparlos.

**La corrección: comparar a maduración fija.** Dos reglas, las dos necesarias:

1. Mirar solamente los **primeros N meses de vida** de cada caso, no toda su historia.
2. Incluir solamente los casos que **ya cumplieron esos N meses**. Los más nuevos quedan afuera, y hay
   que decir que quedaron afuera.

In [ ]:
# Cinco cohortes de una cartera. Datos inventados a propósito, con dos cosas dentro:
# las cohortes nuevas tuvieron menos tiempo de vida Y son de peor calidad.
COHORTES = pd.DataFrame({
    "cohorte":        ["2023 Q1", "2023 Q2", "2023 Q3", "2023 Q4", "2024 Q1"],
    "meses_de_vida":  [14, 11, 8, 5, 2],
    "riesgo_mensual": [0.050, 0.055, 0.070, 0.080, 0.085],
})
CLIENTES_POR_COHORTE = 800
MESES_DE_MADURACION = 3

generador = np.random.default_rng(2026)
filas = []
for fila in COHORTES.itertuples():
    # mes en que cada cliente se atrasa por primera vez
    mes_del_atraso = generador.geometric(fila.riesgo_mensual, CLIENTES_POR_COHORTE)
    filas.append(pd.DataFrame({"cohorte": fila.cohorte,
                               "meses_de_vida": fila.meses_de_vida,
                               "mes_del_atraso": mes_del_atraso}))
clientes = pd.concat(filas, ignore_index=True)

clientes["se_atraso_alguna_vez"] = clientes["mes_del_atraso"] <= clientes["meses_de_vida"]
clientes["se_atraso_en_los_primeros_meses"] = clientes["mes_del_atraso"] <= MESES_DE_MADURACION

# Forma 1: "se atrasó alguna vez", sin corregir por maduración
sin_corregir = clientes.groupby("cohorte").agg(
    clientes=("cohorte", "size"),
    meses_observados=("meses_de_vida", "first"),
    mora_pct=("se_atraso_alguna_vez", lambda columna: columna.mean() * 100),
)
print("FORMA 1: se atrasó alguna vez (sin corregir)")
display(sin_corregir.round(1))

# Forma 2: a maduración fija, y solo las cohortes que ya la alcanzaron
maduras = clientes[clientes["meses_de_vida"] >= MESES_DE_MADURACION]
a_maduracion_fija = maduras.groupby("cohorte").agg(
    clientes=("cohorte", "size"),
    mora_pct=("se_atraso_en_los_primeros_meses", lambda columna: columna.mean() * 100),
)
print(f"\nFORMA 2: se atrasó en los primeros {MESES_DE_MADURACION} meses de vida")
display(a_maduracion_fija.round(1))
excluidas = sorted(set(clientes["cohorte"]) - set(maduras["cohorte"]))
print(f"\nCohortes excluidas por no tener {MESES_DE_MADURACION} meses cumplidos: {excluidas}")

La censura aparece **cada vez que se comparan grupos con distinto tiempo de exposición**. La pregunta de
control es una sola: *¿todos los grupos tuvieron la misma oportunidad de que les pase esto?*